# Line searches

A line search picks how far to move along a direction that is already known to
go downhill. MOpt ships three, and they all share one contract —
`(f, x, d, grad_f) -> (num_iter, eta)` — so they are interchangeable, both on
their own and as the `line_search=` argument of a solver.

In [1]:
import numpy as np
from functools import partial

from mopt.nonlinear import armijo, wolfe, bracketing_wolfe

## A hand-checkable example

Minimize $f(x) = x^T Q x$ from $x_0$ along the fixed direction $d_0$. The
directional derivative $\nabla f(x_0)^T d_0$ is negative, so $d_0$ is a
descent direction and every search below is entitled to run.

In [2]:
Q = np.array([[1.87, -0.21],
              [-0.21, 1.79]])

f = lambda x: x @ Q @ x
grad_f = lambda x: 2.0 * (Q @ x)

x0 = np.array([-2.0, 3.0])
d0 = np.array([7.95, -13.09])

print(f"grad f(x0) = {grad_f(x0)}")
print(f"slope      = {grad_f(x0) @ d0:.4f}   (negative: a descent direction)")

grad f(x0) = [-8.74 11.58]
slope      = -221.0652   (negative: a descent direction)


In [3]:
searches = [
    ("armijo",           partial(armijo, f, x0, d0, grad_f)),
    ("wolfe (weak)",     partial(wolfe, f, x0, d0, grad_f, types="weak")),
    ("wolfe (strong)",   partial(wolfe, f, x0, d0, grad_f, types="strong")),
    ("bracketing_wolfe", partial(bracketing_wolfe, f, x0, d0, grad_f)),
]

for name, run in searches:
    num_iter, eta = run()
    x1 = x0 + eta * d0
    print(f"{name:18} eta = {eta:.6f}   f: {f(x0):7.3f} -> {f(x1):7.3f}   ({num_iter} iters)")

armijo             eta = 0.377255   f:  26.110 ->   9.405   (7 iters)
wolfe (weak)       eta = 0.377255   f:  26.110 ->   9.405   (7 iters)
wolfe (strong)     eta = 0.248423   f:  26.110 ->   0.112   (10 iters)
bracketing_wolfe   eta = 0.250000   f:  26.110 ->   0.132   (3 iters)


## Why the accepted steps differ

Armijo only asks for *sufficient decrease*, so it stops at the first step that
buys enough. The Wolfe conditions add a curvature test on the new slope
$s(\eta) = \nabla f(x_0 + \eta d_0)^T d_0$:

- **weak**: $s(\eta) \ge \sigma\, s(0)$ — tolerates overshooting the valley floor
- **strong**: $|s(\eta)| \le \sigma\, |s(0)|$ — bounds the slope from both sides

Strong Wolfe therefore rejects the step weak Wolfe accepts here: at
$\eta = 0.377$ the slope has already flipped sign and is steeply positive.

In [4]:
def conditions(eta, gamma=0.14, sigma=0.19):
    """Which conditions actually hold at this step."""
    slope = float(grad_f(x0) @ d0)
    new_slope = float(grad_f(x0 + eta * d0) @ d0)
    return {
        "new slope": new_slope,
        "armijo": bool(f(x0 + eta * d0) <= f(x0) + gamma * eta * slope),
        "weak": bool(new_slope >= sigma * slope),
        "strong": bool(abs(new_slope) <= sigma * abs(slope)),
    }

for name, run in searches:
    _, eta = run()
    c = conditions(eta)
    print(f"{name:18} eta={eta:.4f}  slope={c['new slope']:+9.2f}  "
          f"armijo={c['armijo']!s:5} weak={c['weak']!s:5} strong={c['strong']!s}")

armijo             eta=0.3773  slope=  +132.50  armijo=True  weak=True  strong=False
wolfe (weak)       eta=0.3773  slope=  +132.50  armijo=True  weak=True  strong=False
wolfe (strong)     eta=0.2484  slope=   +11.76  armijo=True  weak=True  strong=True
bracketing_wolfe   eta=0.2500  slope=   +13.24  armijo=True  weak=True  strong=True


## Failure modes are reported, not hidden

A direction that is not a descent direction is a caller error and raises. A
search that cannot find an acceptable step raises too, rather than looping
forever or returning a silently bad step.

In [5]:
try:
    armijo(f, x0, -d0, grad_f)          # -d0 points uphill
except ValueError as exc:
    print("ascent direction ->", exc)

try:
    # a coarse shrink factor steps straight over the acceptance window
    wolfe(f, x0, d0, grad_f, types="strong", eta=1.2, delta=0.5)
except RuntimeError as exc:
    print("window skipped   ->", exc)

ascent direction -> d is not a descent direction: grad_f(x) @ d >= 0
window skipped   -> wolfe (strong): no acceptable step within 100 shrinks


## Where backtracking cannot reach

`armijo` and `wolfe` only ever *shrink* the trial step, so they can never
accept a step larger than the one they start from. `bracketing_wolfe` expands
first and then bisects, which matters when the minimizer lies far out along the
direction.

In [6]:
A = np.eye(2)
g = lambda x: 2.0 * (A @ x)
q = lambda x: float(x @ A @ x)

start = np.array([100.0, 0.0])
d = np.array([-1.0, 0.0])          # the minimum along this line is at eta = 100

try:
    wolfe(q, start, d, g)
except RuntimeError as exc:
    print("wolfe            ->", exc)

num_iter, eta = bracketing_wolfe(q, start, d, g)
print(f"bracketing_wolfe -> eta = {eta:.3f} after {num_iter} iters "
      f"(q: {q(start):.1f} -> {q(start + eta * d):.1f})")

wolfe            -> wolfe (strong): no acceptable step within 100 shrinks
bracketing_wolfe -> eta = 16.000 after 5 iters (q: 10000.0 -> 7056.0)


## Plugging one into a solver

Solvers take a line search as an argument. Anything satisfying the same
contract works, including a `functools.partial` with the constants frozen.

In [7]:
from mopt.nonlinear import GradientDescent, NLPProblem

rosenbrock = lambda x: float(np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1.0 - x[:-1])**2))
problem = NLPProblem(f=rosenbrock, x0=[-1.2, 1.0])

for name, search in [("armijo (default)", armijo),
                     ("wolfe, weak", partial(wolfe, types="weak")),
                     ("bracketing_wolfe", bracketing_wolfe)]:
    result = GradientDescent(line_search=search, max_iter=20_000).solve(problem)
    print(f"{name:18} f = {result.fun:.3e} after {result.n_iter} iters   {result.message}")

armijo (default)   f = 2.989e-13 after 13628 iters   Converged: gradient norm below tol.
wolfe, weak        f = 2.989e-13 after 13628 iters   Converged: gradient norm below tol.


bracketing_wolfe   f = 6.078e-13 after 10950 iters   Converged: gradient norm below tol.